# The Climate is Changing: How is Our Sentiment?

## Exploring Changing Sentiment in Climate Articles: 2013 to 2023

#### This is a notebook for extracting data from XML files. Authorship credit to Rafael Alvarado at the University of Virginia; modifications by Caroline Kranefuss, also at the University of Virginia.

## Corpus Description

9 scientific articles are drawn from PeerJ's publications in the decade spanning 2013 to 2023 (3 from 2013, 3 from 2018, and 3 from 2023). Articles are filtered so as to have 50 or more citations, and are sorted by PeerJ's determination of relevance to the query "climate." The top 3 most relevant articles, after meeting these described criteria, have then been chosen to comprise my corpus.

## Imports

In [2]:
# Imports
import pandas as pd 
import numpy as np 
import glob
from lxml import etree
import os
import sys

# ----File Stitching----
# If not in repo home folder, cd back 
if os.path.basename(os.getcwd()) != "evolving_sentiment_climate":
    os.chdir('..')
# If a file is in /sources/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), '..', 'sources'))
source_dir = "sources"
source_files_paths = glob.glob(f"{source_dir}/*.xml")

## Creating DOC Table

In [3]:
# Iterate through files, grab XML info, and save to a dictionary
docs = []
for i, source_file_path in enumerate(source_files_paths):

    # print(source_file_path)

    # Capture tree, root
    tree = etree.parse(source_file_path)
    root = tree.getroot()
    
    # Capture publication id
    pub_id_el = root.xpath("//front//article-meta//article-id")[1]
    pub_id_str = pub_id_el.text
    # print(pub_id_str)

    # Capture title
    title_el = root.xpath("//front//article-title")[0]
    title_str = " ".join([t.strip() for t in title_el.itertext()])
    # print(i, title_str)

    # Capture temporal elements
    year_el = root.xpath("//front//pub-date/year")[0]
    year_str = year_el.text
    month_el = root.xpath("//front//pub-date/month")[0]
    month_str = month_el.text
    day_el = root.xpath("//front//pub-date/day")[0]
    day_str = day_el.text
    date_str = "-".join([year_str, month_str, day_str])
    # print(date_str)

    # Capture key words
    kwd_els = root.xpath("//kwd")
    kwd_strs = [kwd_el.text for kwd_el in kwd_els]
    # print(kwd_strs)

    # Capture each p section (part of OHCO) within the body
    p_els = root.xpath("//body//p")
    p_strs = []
    for p_el in p_els:
        etree.strip_elements(p_el, "xref", with_tail=False)
        p_str = etree.tostring(p_el, method="text", encoding="unicode")
        p_strs.append(p_str)
    # print(p_strs)

    # Add to dictionary
    docs.append({
        'doc_id': pub_id_str,
        'doc_title': title_str,
        'doc_date': date_str,
        'doc_content': " ".join(p_strs),
        'doc_kws': kwd_strs,
        'doc_file_path': source_file_path
    })

# Convert dictionary to data frame
DOC = pd.DataFrame(docs) 
del(docs)

In [4]:
DOC

,doc_id,doc_title,doc_date,doc_content,doc_kws,doc_file_path
0,10.7717/peerj.183,Citations and the h index of soil researchers ...,2013-10-22,Scientific impact measures are increasingly be...,"[Soil science, Bibliometrics, None, Impact fac...",sources/Citations and the h index of soil rese...
1,10.7717/peerj.6030,Structural plasticity in root-fungal symbioses...,2018-12-4,Early terrestrial plants moved from aquatic en...,"[Mycorrhiza, Endophyte, Symbiosis, Root coloni...",sources/Structural plasticity in root-fungal s...
2,10.7717/peerj.151,Typhoon damage on a shallow mesophotic reef in...,2013-9-3,Typhoon damage from direct physical disturbanc...,"[Coral reef, Mesophotic, Typhoon damage, Succe...",sources/Typhoon damage on a shallow mesophotic...
3,10.7717/peerj.163,Microgeographic maladaptive performance and de...,2013-9-17,Identifying informative scales and levels of b...,"[Microgeographic differentiation, Adaptation a...",sources/Microgeographic maladaptive performanc...
4,10.7717/peerj-cs.1306,Artificial intelligence-assisted air quality m...,2023-5-24,Our environment has been greatly impacted by r...,"[AI, Air quality monitoring, Smart cities, Sus...",sources/Artificial intelligence-assisted air q...
5,10.7717/peerj.5885,Effects of waste stream combinations from brew...,2018-11-28,The United Nations figures project global huma...,"[None, Protein quality, Net energy, Mass reari...",sources/Effects of waste stream combinations f...
6,10.7717/peerj.5827,Seasonal and predator-prey effects on circadia...,2018-11-21,Animal activity patterns are influenced by a v...,"[Temporal co-occurrence, Mammal species, Circa...",sources/Seasonal and predator-prey effects on ...
7,10.7717/peerj-cs.1657,Integration of federated learning with IoT for...,2023-12-6,Nobody wants unauthorized access to their data...,"[AI, Smart grid, Federated learning, Internet ...",sources/Integration of federated learning with...
8,10.7717/peerj-cs.1705,Blockchain technology and application: an over...,2023-11-29,Blockchain is a tamper-proof distributed ledge...,"[Blockchain technology, IoT, Consensus mechani...",sources/Blockchain technology and application-...


## Creating TOKEN Table

In [5]:
TOKEN = DOC.doc_content.str.split(expand=True).stack().to_frame("token_str")
TOKEN.index.names = ['doc_id','token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z]+", "", regex=True)
TOKEN = TOKEN[~(TOKEN.term_str == "")]
TOKEN

token_str      term_str
doc_id token_num                            
0      0            Scientific    scientific
       1                impact        impact
       2              measures      measures
       3                   are           are
       4          increasingly  increasingly
...                        ...           ...
8      15801               its           its
       15802         continued     continued
       15803          progress      progress
       15804               and           and
       15805          success.       success

[139308 rows x 2 columns]

Make year, (month?) doc_id index. Split by para, sent, token. Goal; go from doc table to token table with proper index so i  can grouop tokens.

NLTK and Pre-Processing Week 4 - pre-processing pipeline., Salient part import NLTK, import some of these resources in upper cells (under setup - cells 2 and 3 the imports) - will have gotten to point where I have LIB table with documents in them. If I look at the function called tokenize_source, I can modify that to work with XML (he's done that for me maybe?) Can chunk db by paras and sentences.

If paras are divided by newlines? May just jump directly to sentences. 

UVA DS 5001: M04 Pipeline 2 skip to tokenize_source - beginning does what he sent me (importing file, etc) and I need to go to parsing into sentences - copy last half of that function and instead of PARAS use DOC.doc_content

He did crude tokenization because it's not grabbing the sentences. Do this from the M04 nb instead:
SENTS = DOC.doc_content.apply(lambda function).stack
SENTS.index.names = ['doc_id', 'sent_id]
...
TOKENS.index.names = SENTS.index.names + ['token_num]



Make 